
Task 0: Setting up Ollama

a) Set up Ollama and connect to it using either openAI's API or Ollama's own API.

b) Load the 270m parameter version of the gemma3 model and test it with any prompt.

c) Load the 4b parameter version of the gemma3 and test it with any prompt. If running the 4b version is too slow, you can use the 1b version instead.


In [1]:
#a Set up Ollama and connect to it using either openAI's API or Ollama's own API.
!pip install openai



In [22]:
from openai import OpenAI

# Connect to local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  
)

def ask_llm(model, prompt):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test connection (any model you already have)
print(ask_llm("gemma3:270m", "What is sentiment analysis? Answer in one sentence."))



Sentiment analysis is the process of identifying and classifying the emotional tone or attitude expressed in text or data, typically expressed through textual analysis.



In [4]:
#b Load the 270m parameter version of the gemma3 model and test it with any prompt.
MODEL = "gemma3:270m"

print(ask_llm("gemma3:270m","What is sentiment analysis? Answer in one sentence."))

Sentiment analysis is the process of identifying and categorizing the emotional tone or attitude expressed in text, text-based data, or other textual information.



In [23]:
#c Load the 4b parameter version of the gemma3 and test it with any prompt. If running the 4b version is too slow, you can use the 1b version instead.

MODEL = "gemma3:1b"

print(ask_llm("gemma3:1b", "Explain sentiment analysis in one sentence."))

Sentiment analysis is the process of determining the emotional tone or attitude expressed in a piece of text, such as a review or tweet.


Task 1: Text classification with Ollama

a) Make a function for classifying emails (based on the headlines) as spam, work or unknown. The function should return only the classification and nothing else. 

b) Use the smaller gemma3 (270m) to classify the emails using the function created in part a. 

c) Use larger gemma3 (4b) to classify the emails using the function created in part a). In a separate markdown cell, write a brief comment comparing the results of parts b) and c). 

d) Write a script that repeats b) and c) 3 times, storing the results for both models separately. For both models, put the results as columns into a new DataFrame that also contains the headlines so that it is easy to compare how the output varied across runs for both models. Comment if there were differences and explain why this happened. 

In [7]:
#a Make a function for classifying emails (based on the headlines) as spam, work or unknown. The function should return only the classification and nothing else. 

from openai import OpenAI
import pandas as pd

MODEL = "llama3.2"

# Connect to the local Ollama server
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama"  # required by the library but ignored by Ollama
)

def ask_llm(prompt):
    """Send a prompt to the local LLM and return the response text."""
    response = ollama_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# Load dataset
emails = pd.read_csv("data/emails.csv", sep=";")

CATEGORIES = ["spam", "work", "unknown"]

def classify_email(headline):

    categories_str = ", ".join(CATEGORIES)

    prompt = f"""
Classify the following email headline into exactly one of these categories:
{categories_str}

Respond with only the category name, nothing else.

Headline: {headline}
"""

    result = ask_llm(prompt)

    return result.strip()


# Test on first email
print(emails.loc[0, "headline"])
print()
print(classify_email(emails.loc[0, "headline"]))

# Test on second email
print(emails.loc[1, "headline"])
print()
print(classify_email(emails.loc[1, "headline"]))


URGENT: Your account will be suspended within 24 hours

spam
Congratulations! You have won a 1000€ gift card, claim now

spam


In [10]:
#b Use the smaller gemma3 (270m) to classify the emails using the function created in part a

MODEL_270M = "gemma3:270m"

results = []

for headline in emails["headline"]:
    label = classify_email(headline)
    results.append(label)

emails["gemma3_270m"] = results

display(emails[["headline", "gemma3_270m"]])

,headline,gemma3_270m
0,URGENT: Your account will be suspended within ...,spam
1,Congratulations! You have won a 1000€ gift car...,spam
2,Hot singles in your area are waiting to meet y...,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam
4,Meeting agenda for Thursday's project review,work
5,"Q3 budget report attached, please review by Fr...",work
6,Reminder: Annual performance review scheduled ...,work
7,"Updated draft of the manuscript, comments welcome",work
8,Quick question about last week,work
9,Following up,work


In [12]:
#c Use larger gemma3 (4b) to classify the emails using the function created in part a). In separate markdown cell, write a brief comment comparing the results of parts b) and c). 

MODEL_4B = "gemma3:4b"

results_4b = []

for headline in emails["headline"]:
    label = classify_email(headline)
    results_4b.append(label)

emails["gemma3_4b"] = results_4b

display(emails[["headline", "gemma3_270m", "gemma3_4b"]])

,headline,gemma3_270m,gemma3_4b
0,URGENT: Your account will be suspended within ...,spam,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,Spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,Spam
4,Meeting agenda for Thursday's project review,work,work
5,"Q3 budget report attached, please review by Fr...",work,work
6,Reminder: Annual performance review scheduled ...,work,work
7,"Updated draft of the manuscript, comments welcome",work,work
8,Quick question about last week,work,unknown
9,Following up,work,work


#comment

The models gemma3:270m and gemma3:4b perform similarly on the most spam-related emails. The "following up" email from the lighter model gemma3:270m could not be identified as a work email compared to the heavier gemma3:4 b. Additionally, "Are you available?" related to work matters gemma3:270m marked as spam, while heavie gemma3:4b labeled as work, meaning that it could be more accurate in classification tasks. 

In [17]:
#d Write a script that repeats b) and c) 3 times, storing the results for both models separately. For both models, put the results as columns into a new DataFrame that also contains the headlines so that it is easy to compare how the output varied across runs for both models. Comment if there were differences and explain why this happened. 

MODEL_270M = "gemma3:270m"
MODEL_4B = "gemma3:4b"

# puting result dataframe including headline
results_df = pd.DataFrame()
results_df["headline"] = emails["headline"]

# storing runs for both models
for i in range(3):

    # 270M runs
    col_270 = f"270m_run_{i+1}"
    results_df[col_270] = [
        classify_email(h) for h in emails["headline"]
    ]

    # 4B runs
    col_4b = f"4b_run_{i+1}"
    results_df[col_4b] = [
        classify_email(h) for h in emails["headline"]
    ]

# show result table
display(results_df)

,headline,270m_run_1,4b_run_1,270m_run_2,4b_run_2,270m_run_3,4b_run_3
0,URGENT: Your account will be suspended within ...,spam,spam,spam,spam,spam,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam,Spam,spam,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,spam,spam,spam,spam,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,spam,spam,spam,spam,spam
4,Meeting agenda for Thursday's project review,work,work,work,work,work,work
5,"Q3 budget report attached, please review by Fr...",work,work,work,work,work,work
6,Reminder: Annual performance review scheduled ...,work,work,work,work,work,work
7,"Updated draft of the manuscript, comments welcome",work,work,work,work,work,work
8,Quick question about last week,unknown,unknown,work,work,unknown,work
9,Following up,unknown,work,unknown,unknown,unknown,unknown


There were some differences in the categorization of the same email between the repeated runs, particularly in terms of gemma3:270m. The more vague or abstract and short headlines (e.g., "Following up") were the most difficult to identify for the models, though gemma3:4b was more accurate. Overall, gemma3:4b seems to be more stable and accurate compared to the lighter version. It happens because LLMs are probabilistic and they suit for low-stakes tasks where minor errors are acceptable. 

Task 2: Sentiment analysis with Ollama

The data/news.csv file contains 10 fictional financial news headlines. Use it for all subtasks in this task.

a) Make a function for classifying the texts in the provided dataset based on the topic (earnings, mergers, regulation, macroeconomics) and for determining the sentiment of the news (positive, negative, neutral). The function should return the class and sentiment in JSON format. 

b) Use gemma3 (4b) to classify and provide the sentiment for each row of the provided dataset, inserting them into a new DataFrame that contains both the original headlines as well as topic and sentiment. 

c) Give the same data and prompt to a browser based LLM (e.g. ChatGPT, LeChat, Claude or Gemini) and ask it to provide the topic and sentiment, giving it the same options. Paste the results into a markdown cell. Compare the results of b) and c), which one is more accurate and why? 

In [21]:
#a Make a function for classifying the texts in the provided dataset based on the topic (earnings, mergers, regulation, macroeconomics) and for determining the sentiment of the news (positive, negative, neutral). The function should return the class and sentiment in JSON format.

from openai import OpenAI
import pandas as pd
import json

client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama"
)

MODEL = "llama3.2"

def ask_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content
        


# Load dataset
news = pd.read_csv("data/news.csv", sep=";")

TOPICS = ["earnings", "mergers", "regulation", "macroeconomics"]
SENTIMENTS = ["positive", "negative", "neutral"]

#Classifying headline topics and sentiment
def classify_news(headline):

    topics_str = ", ".join(TOPICS)
    sentiments_str = ", ".join(SENTIMENTS)

    prompt = f"""
Classify the following financial news headline.

Possible topics:
{topics_str}

Possible sentiments:
{sentiments_str}

Return ONLY valid JSON in the following format:
{{"topic":"earnings","sentiment":"positive"}}

Headline: {headline}
"""

    result = ask_llm(prompt)

    try:
        parsed = json.loads(result)
        return parsed

    except:
        return {
            "topic": "unknown",
            "sentiment": "unknown"
        }


# Test on first headline
print(news.loc[0, "headline"])
print()
print(classify_news(news.loc[0, "headline"]))

Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%

{'topic': 'earnings', 'sentiment': 'positive'}


In [25]:
#b Use gemma3 (4b) to classify and provide the sentiment for each row of the provided dataset, inserting them into a new DataFrame that contains both the original headlines as well as topic and sentiment

MODEL = "gemma3:1b" #I used gemma3:1b due to the limited capacity of my laptop memory this gemma3:4b was too heavy. 

import json

topics = []
sentiments = []

# Classify all headlines
for headline in news["headline"]:

    result = classify_news(headline)



    topics.append(parsed["topic"])
    sentiments.append(parsed["sentiment"])


# Create new dataframe
news_results = pd.DataFrame()

news_results["headline"] = news["headline"]
news_results["topic"] = topics
news_results["sentiment"] = sentiments

# Show results
display(news_results)

,headline,topic,sentiment
0,Nordion Industries beats Q1 earnings estimates...,mergers,neutral
1,Helvora Pharmaceuticals misses earnings foreca...,mergers,neutral
2,"Aurelis Bank reports steady quarterly profit, ...",mergers,neutral
3,Veridyne Logistics to acquire rival Trantec in...,mergers,neutral
4,Antitrust regulators block proposed merger bet...,mergers,neutral
5,Kestrel Semiconductor confirms early-stage mer...,mergers,neutral
6,New EU AI Act compliance rules expected to rai...,mergers,neutral
7,Finnish FSA grants Norvik Capital expanded lic...,mergers,neutral
8,"Eurozone inflation cools to 2.1%, easing press...",mergers,neutral
9,Rising interest rates weigh on Tessaro Real Es...,mergers,neutral


### Browser-based LLM results (example using ChatGPT)

| Headline | Topic | Sentiment |
|---|---|---|
| Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28% | earnings | positive |
| Helix Pharma to acquire biotech startup in $2B merger deal | mergers | positive |
| Central bank signals possible interest rate cuts later this year | macroeconomics | positive |
| Government announces stricter AI compliance rules for financial institutions | regulation | neutral |
| Retail giant reports declining holiday sales and weaker outlook | earnings | negative |
| Energy firms rally after new climate subsidy package approved | regulation | positive |
| Manufacturing activity contracts for third consecutive month | macroeconomics | negative |
| Telecom merger faces antitrust investigation from EU regulators | mergers | negative |
| Inflation slows unexpectedly as consumer demand weakens | macroeconomics | neutral |
| Tech company shares jump after stronger-than-expected quarterly profits | earnings | positive |

### Comparison of local gemma3 model and browser-based LLM

I applied ChatGPT for this assignment. The browser-based LLM worked much better and uploaded the output much faster.  It also produces more accurate and consistent classifications overall as it is trained on a larger dataset. It is much faster as it doe not use the personal hardware and works via server-side GPU. 


Task 3: Supervised machine learning 

For this task, use a subset of the Bank Marketing dataset, by downloading and importing the bank-additional.csv from the UCI repository. You can find a description of the dataset behind the link.

The goal is to predict whether a prospective customer will subscribe to a term deposit (variable y).

a) Import the dataset and conduct exploratory data analysis on it. 

b) Preprocess the data using the appropriate methods as described in the course materials. 

c) Determine whether this is a classification or regression task. Choose three different machine learning algorithms from scikit-learn and explain briefly why you chose them. For each of the selected algorithsm, train and a model and iteratively adjust the hyperparameters until you no longer manage to improve the performance. 

d) Compare using train, validation and test set split versus using cross-validation. Which one performs better? 

e) Report and evaluate the performance of the models using several of the metrics provided in the course, and explain which model is the best for the task and why. 


In [66]:
# a) Import the dataset and conduct exploratory data analysis on it.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns


# loading dataset from local file (after downloading from UCI)
df = pd.read_csv("data/bank-additional.csv", sep=";")

# Basic overview
display(df.shape)
df.describe()
print(df.head())
print(df.info())
df.isnull().sum()



(4119, 21)

   age          job  marital          education default  housing     loan  \
0   30  blue-collar  married           basic.9y      no      yes       no   
1   39     services   single        high.school      no       no       no   
2   25     services  married        high.school      no      yes       no   
3   38     services  married           basic.9y      no  unknown  unknown   
4   47       admin.  married  university.degree      no      yes       no   

     contact month day_of_week  ...  campaign  pdays  previous     poutcome  \
0   cellular   may         fri  ...         2    999         0  nonexistent   
1  telephone   may         fri  ...         4    999         0  nonexistent   
2  telephone   jun         wed  ...         1    999         0  nonexistent   
3  telephone   jun         fri  ...         3    999         0  nonexistent   
4   cellular   nov         mon  ...         1    999         0  nonexistent   

  emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.e

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [51]:
# b) Preprocess the data using the appropriate methods as described in the course materials.

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Replace missing values coded as "unknown" or "." with NaN
df = df.replace("unknown", np.nan)
df = df.replace(".", np.nan)

# 2. Drop missing values
df = df.dropna()

print("Shape after dropping missing values:", df.shape)

## Separate features and target
X = df.drop('y', axis=1)
y = df['y'].map({'yes': 1, 'no': 0})  # Encode target as binary

# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

Shape after dropping missing values: (3090, 21)


In [67]:

#classification type because variable is y (customer subscribing to the deposit)

#splitting data

from sklearn.model_selection import train_test_split

X = df.drop("y", axis=1)
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [68]:
#RandomForest

"""Random Forest was chosen because it combines multiple decision trees and provides more accurate predictions. It is more effective as it can capture nonlinear relationships that linear models are not able to do""" 

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

best_rf = None
best_score = 0

for n in [50, 100, 200]:
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=n,
            random_state=42
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    score = pipeline.score(X_test, y_test)
    
    if score > best_score:
        best_score = score
        best_rf = pipeline
        
print("Best Random Forest accuracy:", best_score)

pred_rf = best_rf.predict(X_test)

print(confusion_matrix(y_test, pred_rf))
print(classification_report(y_test, pred_rf))

Best Random Forest accuracy: 0.9077669902912622
[[1076   29]
 [  85   46]]
              precision    recall  f1-score   support

          no       0.93      0.97      0.95      1105
         yes       0.61      0.35      0.45       131

    accuracy                           0.91      1236
   macro avg       0.77      0.66      0.70      1236
weighted avg       0.89      0.91      0.90      1236



In [69]:
#K-nearest Neighbours

"""It is based on distance-based classification algorithms, assigning a label to the nearest or similar observation.It helps to reveal patterns in the data""" 

from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

k_values = range(3, 21, 2)

best_score = 0
best_k = None
best_model = None

for k in k_values:
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=k))
    ])
    
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    
    score = accuracy_score(y_test, pred)

    print("k:", k, "Accuracy:", score)

    if score > best_score:
        best_score = score
        best_k = k
        best_model = pipeline

k: 3 Accuracy: 0.8940129449838188
k: 5 Accuracy: 0.8980582524271845
k: 7 Accuracy: 0.8964401294498382
k: 9 Accuracy: 0.901294498381877
k: 11 Accuracy: 0.9045307443365695
k: 13 Accuracy: 0.906957928802589
k: 15 Accuracy: 0.9053398058252428
k: 17 Accuracy: 0.9101941747572816
k: 19 Accuracy: 0.9061488673139159


In [65]:
#LogisticRegression 
""" It is used in binary classification tasks. It predicts the probability that an observation belongs to a particular class using a logistic function. It is easy to interpret, and performs well on structured datasets with clearly defined relationships between variables.
It is easy to interpret."""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# different hyperparameter values
C_values = [0.01, 0.1, 1, 10, 100]

best_score = 0
best_model = None
best_c = None

# iterate through different C values
for c in C_values:
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(
            C=c,
            max_iter=1000,
            random_state=42
        ))
    ])
    
    # train model
    pipeline.fit(X_train, y_train)
    
    # predictions
    pred = pipeline.predict(X_test)
    
    # accuracy
    score = accuracy_score(y_test, pred)
    
    print("C:", c, "Accuracy:", score)
    
    # save best model
    if score > best_score:
        best_score = score
        best_model = pipeline
        best_c = c

# final evaluation
print("\nBest C:", best_c)
print("Best Accuracy:", best_score)

final_pred = best_model.predict(X_test)

print(confusion_matrix(y_test, final_pred))
print(classification_report(y_test, final_pred))

C: 0.01 Accuracy: 0.9158576051779935
C: 0.1 Accuracy: 0.9093851132686084
C: 1 Accuracy: 0.9037216828478964
C: 10 Accuracy: 0.9021035598705501
C: 100 Accuracy: 0.9029126213592233

Best C: 0.01
Best Accuracy: 0.9158576051779935
[[1088   17]
 [  87   44]]
              precision    recall  f1-score   support

          no       0.93      0.98      0.95      1105
         yes       0.72      0.34      0.46       131

    accuracy                           0.92      1236
   macro avg       0.82      0.66      0.71      1236
weighted avg       0.90      0.92      0.90      1236



In [71]:
#d Compare using train, validation and test set split versus using cross-validation. Which one performs better?


from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    KFold
)

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

import pandas as pd

# --------------------------------------------------
# TARGET ENCODING
# --------------------------------------------------

y = df['y'].map({'yes': 1, 'no': 0})
X = df.drop('y', axis=1)

# --------------------------------------------------
# TRAIN / VALIDATION / TEST SPLIT
# --------------------------------------------------

# 80% temp, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 60% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

# --------------------------------------------------
# MODELS
# --------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        C=1,
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=17
    )
}

# --------------------------------------------------
# VALIDATION SET EVALUATION
# --------------------------------------------------

results = []

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # train model
    pipeline.fit(X_train, y_train)

    # predictions
    pred_val = pipeline.predict(X_val)

    # metrics
    accuracy = accuracy_score(y_val, pred_val)
    precision = precision_score(y_val, pred_val)
    recall = recall_score(y_val, pred_val)
    f1 = f1_score(y_val, pred_val)

    # store results
    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

    # print results
    print("\n")
    print("=" * 50)
    print(name)
    print("=" * 50)

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, pred_val))

    print("\nClassification Report:")
    print(classification_report(y_val, pred_val))

# --------------------------------------------------
# RESULTS TABLE
# --------------------------------------------------

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ]
)

print("\n")
print("=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)

print(results_df)

# --------------------------------------------------
# CROSS-VALIDATION
# --------------------------------------------------

print("\n")
print("=" * 50)
print("CROSS-VALIDATION RESULTS")
print("=" * 50)

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=kfold,
        scoring='accuracy'
    )

    print("\n")
    print(name)

    print("Cross-validation scores:")
    print(scores)

    print("Mean CV Accuracy:", scores.mean())

Train shape: (2471, 20)
Validation shape: (824, 20)
Test shape: (824, 20)


Logistic Regression
Accuracy: 0.9162621359223301
Precision: 0.7142857142857143
Recall: 0.3888888888888889
F1-score: 0.5035971223021583

Confusion Matrix:
[[720  14]
 [ 55  35]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95       734
           1       0.71      0.39      0.50        90

    accuracy                           0.92       824
   macro avg       0.82      0.68      0.73       824
weighted avg       0.91      0.92      0.91       824



Random Forest
Accuracy: 0.8992718446601942
Precision: 0.6060606060606061
Recall: 0.2222222222222222
F1-score: 0.3252032520325203

Confusion Matrix:
[[721  13]
 [ 70  20]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.98      0.95       734
           1       0.61      0.22      0.33        90

    accuracy                       

Cross-validation is better as it is less biased and less optimistic about the modes in contrast to train/test. 

#d The model was evaluated based on several metrics such as Accuracy, Precision, Recall, and F1-Score. The best models were the Logistic Regressions as it provided the highest accuracy, precision. At the same time, it is not so effective in detecting the minority class "yes"  in recall. 